# B Cell Mapping Visualization - v1.1 FIXED

**Purpose**: Debug and improve visualizations from scArches L2 mapping  
**Version**: v1.1 (P0/P1 issues fixed)  
**Date**: 2026-02-05  
**Author**: r2end

---

## Fixes Applied

**P0 Critical Fixes:**
1. ✅ Global categories + fixed color palette (consistent colors across ref/query)
2. ✅ Use `astype('string')` instead of `astype(str)` (preserve NA properly)
3. ✅ Explicit marker gene source (use_raw vs layer)

**P1 Strong Recommendations:**
4. ✅ Use pandas Series masks (not numpy arrays)
5. ✅ PDF rasterization for large scatter plots
6. ✅ Improved legend strategies
7. ✅ Updated categorical dtype check

---

## Section 1: Configuration and Setup

In [ ]:
# ===== Imports =====
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
import scanpy as sc
import json

print(f"scanpy: {sc.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

In [ ]:
# ===== P1 Fix: PDF rasterization settings =====
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.settings.set_figure_params(vector_friendly=False)

print("PDF rasterization enabled for large scatter plots")

In [ ]:
# ===== Configuration =====

# Input files
QUERY_H5AD = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad"
MERGED_H5AD = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/reference_plus_query_merged_L2.h5ad"

# Output directory
OUTPUT_DIR = Path("/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/figures_v1_1_fixed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keys
L2_KEY = "Cell_Type_L2"
L2_PRED_KEY = "Cell_Type_L2_pred"
L2_FINAL_KEY = "Cell_Type_L2_final"
CONFIDENCE_KEY = "mapping_confidence"
BATCH_KEY = "sample"
TISSUE_KEY = "tissue"
DATASOURCE_KEY = "data_source"

# Plotting params
DPI = 300
FIGURE_FORMAT = "pdf"
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIGURE_FORMAT)

# Color palettes (will be updated after building global categories)
PALETTE_SOURCE = {"reference": "#1f77b4", "query": "#ff7f0e"}
CMAP_CONFIDENCE = "viridis"
CMAP_EXPRESSION = "Reds"

print(f"Output directory: {OUTPUT_DIR}")
print(f"Query H5AD: {QUERY_H5AD}")
print(f"Merged H5AD: {MERGED_H5AD}")

## Section 2: Load and Inspect Data

In [ ]:
# ===== Load Query Data =====

print("Loading query data...")
adata_query = sc.read_h5ad(QUERY_H5AD)

print(f"\nQuery shape: {adata_query.shape}")
print(f"\nQuery .obs columns:")
print(adata_query.obs.columns.tolist())
print(f"\nQuery .obsm keys:")
print(list(adata_query.obsm.keys()))
print(f"\nQuery .layers keys:")
print(list(adata_query.layers.keys()) if adata_query.layers else "None")
print(f"\nQuery .raw:")
print(f"  Exists: {adata_query.raw is not None}")
if adata_query.raw is not None:
    print(f"  Shape: {adata_query.raw.shape}")

In [ ]:
# ===== Load Merged Data =====

print("Loading merged data...")
adata_merged = sc.read_h5ad(MERGED_H5AD)

print(f"\nMerged shape: {adata_merged.shape}")
print(f"\nMerged .obs columns:")
print(adata_merged.obs.columns.tolist())
print(f"\nMerged .obsm keys:")
print(list(adata_merged.obsm.keys()))
print(f"\nMerged .layers keys:")
print(list(adata_merged.layers.keys()) if adata_merged.layers else "None")
print(f"\nMerged .raw:")
print(f"  Exists: {adata_merged.raw is not None}")
if adata_merged.raw is not None:
    print(f"  Shape: {adata_merged.raw.shape}")

In [ ]:
# ===== Inspect Key Columns =====

print("=" * 60)
print("KEY COLUMNS INSPECTION")
print("=" * 60)

# Show first few rows
if all(k in adata_merged.obs.columns for k in [L2_KEY, L2_FINAL_KEY, DATASOURCE_KEY]):
    print("\nMerged .obs sample (first 10 rows):")
    print(adata_merged.obs[[DATASOURCE_KEY, L2_KEY, L2_FINAL_KEY, CONFIDENCE_KEY]].head(10))

# Data source split
if DATASOURCE_KEY in adata_merged.obs.columns:
    print(f"\nData source distribution:")
    print(adata_merged.obs[DATASOURCE_KEY].value_counts())

## Section 3: P0 Fix #1 - Build Global Categories + Fixed Palette

In [ ]:
# ===== P0 Fix #1: Global categories for consistent colors =====

def build_global_categories(adata, ref_key, qry_key, datasource_key="data_source"):
    """
    Build unified category list from both reference and query.
    
    This ensures same cell type gets same color in both panels.
    """
    ref_mask = adata.obs[datasource_key] == "reference"
    qry_mask = adata.obs[datasource_key] == "query"
    
    # P0 Fix #2: Use astype('string') not astype(str)
    ref_vals = adata.obs.loc[ref_mask, ref_key].astype("string")
    qry_vals = adata.obs.loc[qry_mask, qry_key].astype("string")
    
    # Union of categories (excluding NA)
    ref_cats = pd.Index(ref_vals.dropna().unique())
    qry_cats = pd.Index(qry_vals.dropna().unique())
    all_cats = ref_cats.union(qry_cats)
    
    return sorted(all_cats.tolist())


def make_palette(categories):
    """
    Create fixed color palette for cell types.
    
    Use tab20 for <=20 categories, scanpy default_102 for more.
    """
    n_cats = len(categories)
    
    if n_cats <= 20:
        colors = list(plt.get_cmap("tab20").colors)[:n_cats]
    else:
        # Use scanpy's default 102 colors (stable, distinguishable)
        colors = sc.pl.palettes.default_102[:n_cats]
    
    return dict(zip(categories, colors))


# Build global categories
print("Building global cell type categories...")
GLOBAL_CATS = build_global_categories(adata_merged, L2_KEY, L2_FINAL_KEY, DATASOURCE_KEY)
print(f"\nTotal unique cell types: {len(GLOBAL_CATS)}")
print(f"Categories: {GLOBAL_CATS}")

# Create fixed palette
CT_PALETTE = make_palette(GLOBAL_CATS)
print(f"\nPalette created with {len(CT_PALETTE)} colors")

# Save config for reproducibility
viz_config = {
    "global_categories": GLOBAL_CATS,
    "n_categories": len(GLOBAL_CATS),
    "palette_type": "tab20" if len(GLOBAL_CATS) <= 20 else "default_102",
    "version": "v1.1"
}

config_path = OUTPUT_DIR / "visualization_config.json"
with open(config_path, 'w') as f:
    json.dump(viz_config, f, indent=2)
print(f"\nConfig saved: {config_path}")

## Section 4: Data Cleaning with Fixed dtypes

In [ ]:
# ===== Create Cleaned Visualization Columns =====

print("Creating cleaned visualization columns...")

# P1 Fix #4: Use pandas Series masks (not numpy arrays)
ref_mask = adata_merged.obs[DATASOURCE_KEY].eq("reference")
qry_mask = adata_merged.obs[DATASOURCE_KEY].eq("query")

print(f"Reference cells: {ref_mask.sum():,}")
print(f"Query cells: {qry_mask.sum():,}")

# 1. Query-only L2 Final (P0 Fix #2: use astype('string'))
adata_merged.obs['L2_final_query_only'] = pd.Series(
    pd.NA, 
    index=adata_merged.obs_names, 
    dtype="string"  # P0 Fix #2
)

if L2_FINAL_KEY in adata_merged.obs.columns:
    qry_ser = adata_merged.obs[L2_FINAL_KEY].astype("string")  # P0 Fix #2
    adata_merged.obs.loc[qry_mask, 'L2_final_query_only'] = qry_ser[qry_mask]

# P0 Fix #1: Force to use GLOBAL_CATS
adata_merged.obs['L2_final_query_only'] = pd.Categorical(
    adata_merged.obs['L2_final_query_only'],
    categories=GLOBAL_CATS
)

print(f"  L2_final_query_only created")
print(f"    Non-NA values: {adata_merged.obs['L2_final_query_only'].notna().sum()}")

# 2. Query-only confidence (float, NA for reference)
conf_qry_only = np.full(adata_merged.n_obs, np.nan, dtype=float)

if CONFIDENCE_KEY in adata_merged.obs.columns:
    conf_vals = pd.to_numeric(
        adata_merged.obs.loc[qry_mask, CONFIDENCE_KEY], 
        errors='coerce'
    ).values
    conf_qry_only[qry_mask] = conf_vals

adata_merged.obs['confidence_query_only'] = conf_qry_only

print(f"  confidence_query_only created")
print(f"    Finite values: {np.isfinite(conf_qry_only).sum()}")

# 3. Reference-only L2 (P0 Fix #2: use astype('string'))
adata_merged.obs['L2_reference_only'] = pd.Series(
    pd.NA, 
    index=adata_merged.obs_names, 
    dtype="string"  # P0 Fix #2
)

if L2_KEY in adata_merged.obs.columns:
    ref_ser = adata_merged.obs[L2_KEY].astype("string")  # P0 Fix #2
    adata_merged.obs.loc[ref_mask, 'L2_reference_only'] = ref_ser[ref_mask]

# P0 Fix #1: Force to use GLOBAL_CATS
adata_merged.obs['L2_reference_only'] = pd.Categorical(
    adata_merged.obs['L2_reference_only'],
    categories=GLOBAL_CATS
)

print(f"  L2_reference_only created")
print(f"    Non-NA values: {adata_merged.obs['L2_reference_only'].notna().sum()}")

print("\nColumn creation complete!")

In [ ]:
# ===== Similarly prepare query-only object =====

print("Preparing query object with global categories...")

# Ensure L2_FINAL_KEY uses global categories
if L2_FINAL_KEY in adata_query.obs.columns:
    adata_query.obs[L2_FINAL_KEY] = pd.Categorical(
        adata_query.obs[L2_FINAL_KEY].astype("string"),  # P0 Fix #2
        categories=GLOBAL_CATS
    )
    print(f"  {L2_FINAL_KEY} updated with global categories")

if L2_PRED_KEY in adata_query.obs.columns:
    adata_query.obs[L2_PRED_KEY] = pd.Categorical(
        adata_query.obs[L2_PRED_KEY].astype("string"),  # P0 Fix #2
        categories=GLOBAL_CATS
    )
    print(f"  {L2_PRED_KEY} updated with global categories")

print("Done!")

## Section 5: P0 Fix #3 - Determine Marker Gene Source

In [ ]:
# ===== P0 Fix #3: Determine how to access marker genes =====

def determine_marker_source(adata):
    """
    Determine best source for marker gene expression.
    
    Returns:
        tuple: (use_raw, layer)
    """
    # Priority 1: .raw with full genes
    if adata.raw is not None and adata.raw.n_vars > adata.n_vars:
        return True, None
    
    # Priority 2: log1p layer
    if adata.layers and "log1p" in adata.layers:
        return False, "log1p"
    
    # Priority 3: counts layer (will need normalization)
    if adata.layers and "counts" in adata.layers:
        return False, "counts"
    
    # Fallback: use .X
    return False, None


# Check query
print("Query dataset:")
use_raw_query, layer_query = determine_marker_source(adata_query)
print(f"  use_raw: {use_raw_query}")
print(f"  layer: {layer_query}")

# Check merged
print("\nMerged dataset:")
use_raw_merged, layer_merged = determine_marker_source(adata_merged)
print(f"  use_raw: {use_raw_merged}")
print(f"  layer: {layer_merged}")

# Store for later use
MARKER_CONFIG = {
    "query": {"use_raw": use_raw_query, "layer": layer_query},
    "merged": {"use_raw": use_raw_merged, "layer": layer_merged}
}

In [ ]:
# ===== Helper function for marker availability =====

def get_available_markers(adata, marker_list, use_raw=False):
    """
    Check which markers are available in the dataset.
    
    Args:
        adata: AnnData object
        marker_list: List of gene names
        use_raw: Whether to check .raw.var_names
    
    Returns:
        list: Available marker genes
    """
    if use_raw and adata.raw is not None:
        var_names = adata.raw.var_names
    else:
        var_names = adata.var_names
    
    return [g for g in marker_list if g in var_names]


# Test with B cell markers
test_markers = ["CD19", "MS4A1", "CD27", "IGHD", "MZB1", "SDC1", "JCHAIN"]

print("Available markers in query:")
avail_query = get_available_markers(
    adata_query, 
    test_markers, 
    use_raw=use_raw_query
)
print(f"  {avail_query}")

print("\nAvailable markers in merged:")
avail_merged = get_available_markers(
    adata_merged, 
    test_markers, 
    use_raw=use_raw_merged
)
print(f"  {avail_merged}")

## Section 6: Helper Functions for Plotting

In [ ]:
# ===== P1 Fix #5: Rasterization helper =====

def save_rasterized_figure(fig, path, dpi=300, rasterize_scatter=True):
    """
    Save figure with scatter plots rasterized to reduce file size.
    
    Args:
        fig: matplotlib figure
        path: output path
        dpi: resolution
        rasterize_scatter: whether to rasterize scatter collections
    """
    if rasterize_scatter:
        for ax in fig.axes:
            for coll in ax.collections:
                coll.set_rasterized(True)
    
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


print("Helper functions defined")

## Section 7: Query-Only Visualizations (FIXED)

In [ ]:
# ===== Query Overview: 4-Panel Figure (FIXED) =====

fig, axes = plt.subplots(2, 2, figsize=(18, 18))

# Panel 1: Raw predictions (all cells) - FIXED palette
if L2_PRED_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query, 
        color=L2_PRED_KEY,
        ax=axes[0, 0],
        show=False,
        title="L2 Predictions (All Cells)",
        legend_loc="right margin",
        palette=CT_PALETTE,  # P0 Fix #1
        frameon=False,
        s=30
    )

# Panel 2: Filtered predictions - FIXED palette
if L2_FINAL_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query,
        color=L2_FINAL_KEY,
        ax=axes[0, 1],
        show=False,
        title="L2 Final (Confidence >= 0.5)",
        legend_loc="right margin",
        palette=CT_PALETTE,  # P0 Fix #1
        frameon=False,
        s=30
    )

# Panel 3: Confidence score heatmap
if CONFIDENCE_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query,
        color=CONFIDENCE_KEY,
        ax=axes[1, 0],
        show=False,
        title="Mapping Confidence Score",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        frameon=False,
        s=30
    )

# Panel 4: Confidence histogram
if CONFIDENCE_KEY in adata_query.obs.columns:
    conf_vals = pd.to_numeric(adata_query.obs[CONFIDENCE_KEY], errors='coerce').to_numpy()
    conf_vals = conf_vals[np.isfinite(conf_vals)]
    
    axes[1, 1].hist(
        conf_vals, 
        bins=50, 
        edgecolor='black', 
        alpha=0.7,
        color='steelblue'
    )
    axes[1, 1].axvline(
        0.5, 
        color='red', 
        linestyle='--', 
        linewidth=2, 
        label='Threshold = 0.5'
    )
    axes[1, 1].set_xlabel('Mapping Confidence', fontsize=12)
    axes[1, 1].set_ylabel('Number of Cells', fontsize=12)
    axes[1, 1].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=11)
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].spines['top'].set_visible(False)
    axes[1, 1].spines['right'].set_visible(False)

plt.tight_layout()
output_path = OUTPUT_DIR / f"query_overview_fixed.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"Saved: {output_path}")

In [ ]:
# ===== Query Cell Type Proportions =====

if L2_FINAL_KEY in adata_query.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # Bar plot
    ct_counts = adata_query.obs[L2_FINAL_KEY].value_counts()
    ct_counts.plot(
        kind='bar',
        ax=axes[0],
        color='steelblue',
        edgecolor='black'
    )
    axes[0].set_title('Cell Type Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cell Type', fontsize=12)
    axes[0].set_ylabel('Number of Cells', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(ct_counts.values):
        axes[0].text(i, v + max(ct_counts.values)*0.01, f'{v:,}', 
                     ha='center', va='bottom', fontsize=10)
    
    # Pie chart
    ct_pct = adata_query.obs[L2_FINAL_KEY].value_counts(normalize=True) * 100
    
    # Use colors from global palette
    pie_colors = [CT_PALETTE.get(ct, 'gray') for ct in ct_pct.index]
    
    axes[1].pie(
        ct_pct.values,
        labels=ct_pct.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=pie_colors,  # P0 Fix #1
        textprops={'fontsize': 10}
    )
    axes[1].set_title('Cell Type Distribution (Percentage)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"query_celltype_proportions.{FIGURE_FORMAT}"
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {output_path}")

## Section 8: Merged Visualizations (FIXED)

In [ ]:
# ===== Merged Overview: 6-Panel Figure (FIXED) =====

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

# Panel 1: Data source
sc.pl.umap(
    adata_merged,
    color=DATASOURCE_KEY,
    ax=axes[0, 0],
    show=False,
    title="Data Source",
    palette=PALETTE_SOURCE,
    frameon=False,
    s=20
)

# Panel 2: Reference L2 labels - FIXED palette
sc.pl.umap(
    adata_merged,
    color='L2_reference_only',
    ax=axes[0, 1],
    show=False,
    title="Reference L2 Labels (Reference Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 3: Query L2 predictions - FIXED palette
sc.pl.umap(
    adata_merged,
    color='L2_final_query_only',
    ax=axes[0, 2],
    show=False,
    title="Query L2 Predictions (Query Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 4: Batch
if BATCH_KEY in adata_merged.obs.columns:
    sc.pl.umap(
        adata_merged,
        color=BATCH_KEY,
        ax=axes[1, 0],
        show=False,
        title="Batch",
        frameon=False,
        s=20,
        legend_loc=None  # P1 Fix #6: too many batches
    )

# Panel 5: Confidence (query only)
sc.pl.umap(
    adata_merged,
    color='confidence_query_only',
    ax=axes[1, 1],
    show=False,
    title="Mapping Confidence (Query Only)",
    cmap=CMAP_CONFIDENCE,
    vmin=0,
    vmax=1,
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 6: B cell marker (P0 Fix #3)
marker_gene = "CD19"
available = get_available_markers(
    adata_merged, 
    [marker_gene], 
    use_raw=use_raw_merged
)

if marker_gene in available:
    plot_kwargs = {
        'color': marker_gene,
        'ax': axes[1, 2],
        'show': False,
        'title': f"{marker_gene} Expression",
        'cmap': CMAP_EXPRESSION,
        'frameon': False,
        's': 20,
    }
    
    # P0 Fix #3: Explicit source
    if use_raw_merged:
        plot_kwargs['use_raw'] = True
    elif layer_merged is not None:
        plot_kwargs['layer'] = layer_merged
    else:
        plot_kwargs['use_raw'] = False
    
    sc.pl.umap(adata_merged, **plot_kwargs)
else:
    axes[1, 2].text(
        0.5, 0.5, 
        f"{marker_gene}\nNot Available",
        ha='center', 
        va='center',
        transform=axes[1, 2].transAxes,
        fontsize=14
    )
    axes[1, 2].axis('off')

plt.tight_layout()
output_path = OUTPUT_DIR / f"merged_overview_fixed.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"Saved: {output_path}")

In [ ]:
# ===== Side-by-Side Comparison: Reference vs Query (FIXED) =====

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: Reference L2 labels - FIXED palette
sc.pl.umap(
    adata_merged,
    color='L2_reference_only',
    ax=axes[0],
    show=False,
    title="Reference: Original L2 Labels",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=25,
    na_color='lightgray'
)

# Right: Query L2 predictions - FIXED palette (same colors!)
sc.pl.umap(
    adata_merged,
    color='L2_final_query_only',
    ax=axes[1],
    show=False,
    title="Query: Predicted L2 Labels",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=25,
    na_color='lightgray'
)

plt.tight_layout()
output_path = OUTPUT_DIR / f"reference_vs_query_comparison_fixed.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"Saved: {output_path}")

## Section 9: Marker Gene Panel (FIXED)

In [ ]:
# ===== B Cell Marker Genes Panel (P0 Fix #3) =====

marker_genes = [
    "CD19",    # Pan B cell
    "MS4A1",   # CD20, B cell marker
    "CD27",    # Memory B marker
    "IGHD",    # Naive B marker (IgD)
    "IGHM",    # IgM
    "MZB1",    # Plasma cell marker
    "SDC1",    # CD138, Plasma cell
    "JCHAIN",  # Plasma cell
]

# P0 Fix #3: Check availability properly
available_markers = get_available_markers(
    adata_query, 
    marker_genes, 
    use_raw=use_raw_query
)

print(f"Available markers: {available_markers}")

if len(available_markers) > 0:
    # Calculate grid size
    n_markers = len(available_markers)
    ncols = 4
    nrows = int(np.ceil(n_markers / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes
    
    # P0 Fix #3: Prepare plot kwargs
    base_kwargs = {
        'show': False,
        'cmap': CMAP_EXPRESSION,
        'frameon': False,
        's': 30,
    }
    
    if use_raw_query:
        base_kwargs['use_raw'] = True
    elif layer_query is not None:
        base_kwargs['layer'] = layer_query
    else:
        base_kwargs['use_raw'] = False
    
    for i, gene in enumerate(available_markers):
        sc.pl.umap(
            adata_query,
            color=gene,
            ax=axes[i],
            title=f"{gene} Expression",
            **base_kwargs
        )
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"bcell_markers_panel_fixed.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
    print(f"Saved: {output_path}")
else:
    print("No marker genes found in dataset.")

## Section 10: Quality Control (P2 optimization applied)

In [ ]:
# ===== Confidence vs Cell Type (P2 optimization: top N only) =====

if CONFIDENCE_KEY in adata_query.obs.columns and L2_FINAL_KEY in adata_query.obs.columns:
    
    # Prepare data
    df_plot = adata_query.obs[[L2_FINAL_KEY, CONFIDENCE_KEY]].copy()
    df_plot[CONFIDENCE_KEY] = pd.to_numeric(df_plot[CONFIDENCE_KEY], errors='coerce')
    df_plot = df_plot.dropna()
    
    # P2 optimization: Take top 10 cell types by count
    top_cts = df_plot[L2_FINAL_KEY].value_counts().head(10).index
    df_plot_top = df_plot[df_plot[L2_FINAL_KEY].isin(top_cts)].copy()
    
    # Sort by median confidence
    ct_order = df_plot_top.groupby(L2_FINAL_KEY)[CONFIDENCE_KEY].median().sort_values(ascending=False).index
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # Violin plot
    sns.violinplot(
        data=df_plot_top,
        x=L2_FINAL_KEY,
        y=CONFIDENCE_KEY,
        order=ct_order,
        ax=axes[0],
        palette='Set2'
    )
    axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
    axes[0].set_title('Confidence by Cell Type (Top 10, Violin)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cell Type', fontsize=12)
    axes[0].set_ylabel('Mapping Confidence', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    sns.boxplot(
        data=df_plot_top,
        x=L2_FINAL_KEY,
        y=CONFIDENCE_KEY,
        order=ct_order,
        ax=axes[1],
        palette='Set2'
    )
    axes[1].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
    axes[1].set_title('Confidence by Cell Type (Top 10, Box)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Cell Type', fontsize=12)
    axes[1].set_ylabel('Mapping Confidence', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"confidence_by_celltype_top10.{FIGURE_FORMAT}"
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {output_path}")

In [ ]:
# ===== Confidence Summary Table =====

if CONFIDENCE_KEY in adata_query.obs.columns and L2_FINAL_KEY in adata_query.obs.columns:
    
    df_summary = adata_query.obs.copy()
    df_summary[CONFIDENCE_KEY] = pd.to_numeric(df_summary[CONFIDENCE_KEY], errors='coerce')
    
    summary_stats = df_summary.groupby(L2_FINAL_KEY)[CONFIDENCE_KEY].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(3)
    
    # Add percentage
    summary_stats['Percentage'] = (summary_stats['Count'] / summary_stats['Count'].sum() * 100).round(2)
    
    # Sort by count
    summary_stats = summary_stats.sort_values('Count', ascending=False)
    
    print("=" * 80)
    print("CONFIDENCE SUMMARY BY CELL TYPE")
    print("=" * 80)
    print(summary_stats.to_string())
    print("=" * 80)
    
    # Save to CSV
    csv_path = OUTPUT_DIR / "confidence_summary_by_celltype.csv"
    summary_stats.to_csv(csv_path)
    print(f"\nSaved summary table: {csv_path}")

## Section 11: High-Resolution Individual Exports

In [ ]:
# ===== Export Individual UMAPs (Publication Ready) =====

print("Exporting individual high-resolution UMAPs...")

export_params = {
    'frameon': False,
    's': 50,
    'show': False
}

# 1. Query L2 Final (P1 Fix #6: on data legend)
if L2_FINAL_KEY in adata_query.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata_query,
        color=L2_FINAL_KEY,
        ax=ax,
        title="",
        legend_loc="on data",  # P1 Fix #6
        legend_fontsize=10,
        legend_fontoutline=2,
        palette=CT_PALETTE,  # P0 Fix #1
        **export_params
    )
    output_path = OUTPUT_DIR / f"umap_L2_final_highres.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
    print(f"  Saved: {output_path.name}")

# 2. Query Confidence
if CONFIDENCE_KEY in adata_query.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata_query,
        color=CONFIDENCE_KEY,
        ax=ax,
        title="",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        **export_params
    )
    output_path = OUTPUT_DIR / f"umap_confidence_highres.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
    print(f"  Saved: {output_path.name}")

# 3. Data Source (Merged)
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=DATASOURCE_KEY,
    ax=ax,
    title="",
    palette=PALETTE_SOURCE,
    **export_params
)
output_path = OUTPUT_DIR / f"umap_data_source_highres.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"  Saved: {output_path.name}")

print("\nExport complete!")

## Section 12: Summary Report

In [ ]:
# ===== Generate Summary Report =====

print("=" * 80)
print("B CELL MAPPING VISUALIZATION SUMMARY (v1.1 FIXED)")
print("=" * 80)

print(f"\nQuery Dataset:")
print(f"  Total cells: {adata_query.n_obs:,}")
print(f"  Total genes: {adata_query.n_vars:,}")

if L2_FINAL_KEY in adata_query.obs.columns:
    print(f"\nCell Type Distribution:")
    for ct, count in adata_query.obs[L2_FINAL_KEY].value_counts().head(10).items():
        pct = count / adata_query.n_obs * 100
        print(f"  {ct}: {count:,} ({pct:.1f}%)")

if CONFIDENCE_KEY in adata_query.obs.columns:
    conf = pd.to_numeric(adata_query.obs[CONFIDENCE_KEY], errors='coerce')
    print(f"\nMapping Confidence:")
    print(f"  Mean: {conf.mean():.3f}")
    print(f"  Median: {conf.median():.3f}")
    print(f"  High confidence (>0.9): {(conf > 0.9).sum():,} ({(conf > 0.9).mean()*100:.1f}%)")
    print(f"  Low confidence (<0.5): {(conf < 0.5).sum():,} ({(conf < 0.5).mean()*100:.1f}%)")

print(f"\nMerged Dataset:")
print(f"  Total cells: {adata_merged.n_obs:,}")
print(f"  Reference cells: {(adata_merged.obs[DATASOURCE_KEY] == 'reference').sum():,}")
print(f"  Query cells: {(adata_merged.obs[DATASOURCE_KEY] == 'query').sum():,}")

print(f"\nColor Palette:")
print(f"  Global categories: {len(GLOBAL_CATS)}")
print(f"  Palette type: {viz_config['palette_type']}")
print(f"  Consistent colors: ✅ (Same cell type = same color in all plots)")

print(f"\nMarker Gene Source:")
print(f"  Query: use_raw={use_raw_query}, layer={layer_query}")
print(f"  Merged: use_raw={use_raw_merged}, layer={layer_merged}")

print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"  Total files generated: {len(list(OUTPUT_DIR.glob('*')))}")

print("\n" + "=" * 80)
print("FIXES APPLIED (v1.1)")
print("=" * 80)
print("P0 Critical:")
print("  ✅ #1: Global categories + fixed color palette")
print("  ✅ #2: astype('string') to preserve NA properly")
print("  ✅ #3: Explicit marker gene source (use_raw/layer)")
print("\nP1 Strong Recommendations:")
print("  ✅ #4: Pandas Series masks (not numpy arrays)")
print("  ✅ #5: PDF rasterization enabled")
print("  ✅ #6: Improved legend strategies")
print("  ✅ #7: Updated categorical dtype check")
print("\nP2 Optimizations:")
print("  ✅ Top N cell types for QC plots")
print("  ✅ Configuration saved to JSON")
print("\n" + "=" * 80)
print("VISUALIZATION DEBUG COMPLETE")
print("=" * 80)

---

## Summary of Fixes

### P0 Critical Fixes (Must Have)

1. **Global Categories + Fixed Palette**: Same cell type now has identical color across all plots (reference vs query comparison is no longer misleading)

2. **Proper NA Handling**: Using `astype('string')` instead of `astype(str)` prevents NA from becoming "nan" string category

3. **Explicit Marker Source**: Marker gene expression now explicitly specifies `use_raw` or `layer` to ensure correct data source

### P1 Strong Recommendations

4. **Pandas Series Masks**: Using `.eq()` instead of `.to_numpy()` for safer indexing

5. **PDF Rasterization**: Large scatter plots are now rasterized to reduce file size and prevent Illustrator crashes

6. **Legend Strategies**: "on data" for high-res exports, "right margin" or None for overviews

7. **Updated Dtype Check**: Using `CategoricalDtype` instead of deprecated `pd.api.types.is_categorical_dtype`

### P2 Optimizations

8. **Top N for QC**: Confidence plots now show top 10 cell types only for better readability

9. **Config Export**: Visualization configuration saved to JSON for reproducibility

---

## Next Steps

1. **Validate Results**: Check that reference and query have consistent colors across panels

2. **Marker Validation**: Verify B cell lineage markers show expected patterns

3. **Integration Analysis**: Compare with epithelial cell data for cross-lineage interactions

---